In [ ]:
# ==============================================================================
# 1. INSTALACIÓN DE DEPENDENCIAS (Ejecutar sólo si es necesario)
# ==============================================================================
# !pip install dash pandas plotly geopandas

import dash
from dash import dcc, html, Input, Output
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# ==============================================================================
# 2. PREPROCESAMIENTO DE DATOS (ETL)
# ==============================================================================
print("Cargando y procesando datos... (Esto puede tomar unos segundos)")

# --- 2.1 Datos de Urgencias Respiratorias ---
for year in range(2020, 2026):
    try:
        df = pd.read_csv(f'AtencionesRespiratorias/AtencionesRespiratorias{year}.csv')
        df['Comuna'] = df['ComunaGlosa'] # Estandarizamos nombre
        globals()[f'df_urg_{year}'] = df
    except: 
        globals()[f'df_urg_{year}'] = pd.DataFrame()


df_urgencias = pd.concat([df_urg_2020,df_urg_2021,df_urg_2022,df_urg_2023,df_urg_2024,df_urg_2025], ignore_index=True)
if not df_urgencias.empty:
    df_urgencias['fecha'] = pd.to_datetime(df_urgencias['fecha'], format='%d/%m/%Y', errors='coerce')
    df_urgencias['Mes'] = df_urgencias['fecha'].dt.month
    df_urgencias['Año'] = df_urgencias['fecha'].dt.year

# --- 2.2 Datos de Contaminación PM 2.5 ---
ciudades_pm = ['Osorno', 'Chiguayante', 'Concepcion', 'Temuco', 'Valdivia']
lista_pm = []
for ciudad in ciudades_pm:
    try:
        df_c = pd.read_csv(f'Datos_PM/{ciudad}.csv', sep=';', encoding='latin-1') 
        df_c['Comuna'] = ciudad
        df_c['fecha'] = pd.to_datetime(df_c['FECHA (YYMMDD)'], format='%y%m%d', errors='coerce')
        df_c.rename(columns={'Registros validados': 'PM25'}, inplace=True)
        # Limpieza de valores (comas por puntos)
        df_c['PM25'] = pd.to_numeric(df_c['PM25'].astype(str).str.replace(',', '.'), errors='coerce')
        lista_pm.append(df_c)
    except:
        pass
df_pm25 = pd.concat(lista_pm, ignore_index=True) if lista_pm else pd.DataFrame()

# --- 2.3 Datos de Camas Críticas (REM20) ---
try:
    df_camas = pd.read_csv('Indicadores de Salud REM20/Indicadores REM20.csv')
    # Filtramos por las áreas más críticas si es necesario (ejemplo: UCI/UTI)
    df_camas = df_camas[df_camas['AREA_FUNCIONAL'].str.contains('Intensiv|Crític', case=False, na=False)]
except:
    df_camas = pd.DataFrame()

# Coordenadas estáticas para el mapa de burbujas (Como alternativa rápida al .shp)
coords = {
    'Osorno': {'lat': -40.5739, 'lon': -73.1336},
    'Chiguayante': {'lat': -36.9209, 'lon': -73.0253},
    'Concepcion': {'lat': -36.8201, 'lon': -73.0444}, # Sin tilde por estándar de datos
    'Temuco': {'lat': -38.7359, 'lon': -72.5904},
    'Valdivia': {'lat': -39.8196, 'lon': -73.2452}
}

# ==============================================================================
# 3. INICIALIZACIÓN DE LA APP DASH
# ==============================================================================
app = dash.Dash(__name__)

# Estilos CSS básicos para mantener organización y coherencia (Rúbrica punto 4)
estilo_tarjeta = {
    'backgroundColor': 'white', 'padding': '20px', 'borderRadius': '10px',
    'boxShadow': '0 4px 6px rgba(0,0,0,0.1)', 'marginBottom': '20px'
}

# ==============================================================================
# 4. LAYOUT DEL DASHBOARD
# ==============================================================================
app.layout = html.Div(style={'fontFamily': 'Arial, sans-serif', 'backgroundColor': '#f4f6f9', 'padding': '30px'}, children=[
    
    # ENCABEZADO Y OBJETIVO (Rúbrica Puntos 1 y 3)
    html.Div(style=estilo_tarjeta, children=[
        html.H1("Dashboard: Saturación Hospitalaria, Virus y Calidad del Aire (PM 2.5)", 
                style={'color': '#2c3e50', 'textAlign': 'center'}),
        
        html.Div(style={'backgroundColor': '#e8f4f8', 'padding': '15px', 'borderRadius': '8px', 'borderLeft': '5px solid #3498db'}, children=[
            html.H3("🎯 Objetivo de la Visualización", style={'marginTop': '0', 'color': '#2980b9'}),
            html.P([html.B("Problema: "), "Explorar la relación entre los niveles de contaminación atmosférica (PM 2.5), el aumento de atenciones de urgencia por causas respiratorias y la ocupación de camas críticas en el sur de Chile."]),
            html.P([html.B("Usuario: "), "Analistas de Salud Pública, Gestores Hospitalarios y Seremis de Salud."]),
            html.P([html.B("Contexto de uso: "), "Análisis retrospectivo y toma de decisiones estratégicas para la asignación de recursos y declaratoria de alertas ambientales/sanitarias durante períodos de alta carga viral y contaminación."])
        ])
    ]),

    # CONTROLES Y FILTROS (Inputs)
    html.Div(style=estilo_tarjeta, children=[
        html.H4("⚙️ Filtros Globales", style={'color': '#2c3e50'}),
        html.Div(style={'display': 'flex', 'gap': '20px'}, children=[
            html.Div(style={'flex': '1'}, children=[
                html.Label("Seleccione Año:"),
                dcc.Dropdown(
                    id='filtro-ano',
                    options=[{'label': str(ano), 'value': ano} for ano in [2020, 2024]],
                    value=2020,
                    clearable=False
                )
            ]),
            html.Div(style={'flex': '1'}, children=[
                html.Label("Seleccione Comuna (Zona de Estudio):"),
                dcc.Dropdown(
                    id='filtro-comuna',
                    options=[{'label': c, 'value': c} for c in ciudades_pm],
                    value='Osorno',
                    clearable=False
                )
            ])
        ])
    ]),

    # SECCIÓN 1: MAPA Y COMPOSICIÓN (Gráficos 1 y 2)
    html.Div(style={'display': 'flex', 'gap': '20px'}, children=[
        html.Div(style={**estilo_tarjeta, 'flex': '1'}, children=[
            html.H4("1. Mapa de Focos: Promedio PM 2.5 y Casos", style={'textAlign': 'center'}),
            html.P("Mapa con marcadores (Burbujas) escalado por nivel de contaminación histórico.", style={'fontSize': '12px', 'color': 'gray', 'textAlign': 'center'}),
            dcc.Graph(id='grafico-mapa')
        ]),
        html.Div(style={**estilo_tarjeta, 'flex': '1'}, children=[
            html.H4("2. Composición de Causas Respiratorias", style={'textAlign': 'center'}),
            html.P("Gráfica de composición (Pie Chart) de las principales enfermedades.", style={'fontSize': '12px', 'color': 'gray', 'textAlign': 'center'}),
            dcc.Graph(id='grafico-composicion')
        ])
    ]),

    # SECCIÓN 2: EVOLUCIÓN TEMPORAL COMPARATIVA (Gráfico 3)
    html.Div(style=estilo_tarjeta, children=[
        html.H4("3. Evolución Comparativa: PM 2.5 vs Atenciones Respiratorias", style={'textAlign': 'center'}),
        html.P("Gráfica de series de tiempo doble eje. Permite observar si los peaks de PM 2.5 anteceden a los ingresos a urgencias.", style={'fontSize': '12px', 'color': 'gray', 'textAlign': 'center'}),
        dcc.Graph(id='grafico-serie-tiempo')
    ]),

    # SECCIÓN 3: CORRELACIÓN Y DISTRIBUCIÓN (Gráficos 4, 5 y 6)
    html.Div(style={'display': 'flex', 'gap': '20px'}, children=[
        html.Div(style={**estilo_tarjeta, 'flex': '1'}, children=[
            html.H4("4. Relación PM 2.5 y Volumen de Urgencias", style={'textAlign': 'center'}),
            html.P("Gráfica de correlación (Scatter). Cada punto es una semana.", style={'fontSize': '12px', 'color': 'gray', 'textAlign': 'center'}),
            dcc.Graph(id='grafico-correlacion')
        ]),
        html.Div(style={**estilo_tarjeta, 'flex': '1'}, children=[
            html.H4("5. Distribución de Ocupación UCI/UTI", style={'textAlign': 'center'}),
            html.P("Gráfica de distribución estadística (Boxplot) por mes del Índice Ocupacional.", style={'fontSize': '12px', 'color': 'gray', 'textAlign': 'center'}),
            dcc.Graph(id='grafico-distribucion')
        ]),
        html.Div(style={**estilo_tarjeta, 'flex': '1'}, children=[
            html.H4("6. Tendencia Camas Críticas Ocupadas", style={'textAlign': 'center'}),
            html.P("Gráfica de serie de tiempo simple sobre la ocupación total mensual.", style={'fontSize': '12px', 'color': 'gray', 'textAlign': 'center'}),
            dcc.Graph(id='grafico-camas-serie')
        ])
    ])
])

# ==============================================================================
# 5. CALLBACKS (Lógica Reactiva - Múltiples Inputs/Outputs)
# ==============================================================================
@app.callback(
    [Output('grafico-mapa', 'figure'),
     Output('grafico-composicion', 'figure'),
     Output('grafico-serie-tiempo', 'figure'),
     Output('grafico-correlacion', 'figure'),
     Output('grafico-distribucion', 'figure'),
     Output('grafico-camas-serie', 'figure')],
    [Input('filtro-ano', 'value'),
     Input('filtro-comuna', 'value')]
)
def actualizar_dashboard(ano_sel, comuna_sel):
    
    # -- Filtros de Datos --
    df_u_filt = df_urgencias[(df_urgencias['Año'] == ano_sel) & (df_urgencias['Comuna'] == comuna_sel)] if not df_urgencias.empty else pd.DataFrame()
    df_p_filt = df_pm25[(df_pm25['fecha'].dt.year == ano_sel) & (df_pm25['Comuna'] == comuna_sel)] if not df_pm25.empty else pd.DataFrame()
    df_c_filt = df_camas[(df_camas['PERIODO'] == ano_sel)] if not df_camas.empty else pd.DataFrame() # Camas suele ser regional

    # -------------------------------------------------------------------------
    # 1. Mapa de Marcadores
    # -------------------------------------------------------------------------
    map_data = []
    for c, latlon in coords.items():
        pm_mean = df_pm25[(df_pm25['fecha'].dt.year == ano_sel) & (df_pm25['Comuna'] == c)]['PM25'].mean()
        if pd.isna(pm_mean): pm_mean = 10 # Valor base si no hay datos
        map_data.append({'Comuna': c, 'Lat': latlon['lat'], 'Lon': latlon['lon'], 'PM25 Promedio': pm_mean})
    df_map = pd.DataFrame(map_data)
    
    fig_map = px.scatter_mapbox(df_map, lat="Lat", lon="Lon", size="PM25 Promedio", color="PM25 Promedio",
                                hover_name="Comuna", color_continuous_scale=px.colors.sequential.YlOrRd, size_max=25, zoom=5)
    fig_map.update_layout(mapbox_style="carto-positron", margin={"r":0,"t":0,"l":0,"b":0})

    # -------------------------------------------------------------------------
    # 2. Gráfica de Composición (Pie)
    # -------------------------------------------------------------------------
    if not df_u_filt.empty and 'GlosaCausa' in df_u_filt.columns:
        df_causas = df_u_filt.groupby('GlosaCausa')['Total'].sum().reset_index()
        # Agrupar causas menores
        df_causas = df_causas.sort_values('Total', ascending=False).head(5) 
        fig_comp = px.pie(df_causas, names='GlosaCausa', values='Total', hole=0.4, 
                          color_discrete_sequence=px.colors.qualitative.Pastel)
    else:
        fig_comp = go.Figure().add_annotation(text="Sin datos", showarrow=False)
    fig_comp.update_layout(margin=dict(t=20, b=20, l=20, r=20), showlegend=False) # Leyenda oculta para espacio, labels en hover

    # -------------------------------------------------------------------------
    # 3. Gráfica Comparativa Serie de Tiempo (Doble Eje)
    # -------------------------------------------------------------------------
    fig_ts = go.Figure()
    if not df_p_filt.empty:
        df_p_week = df_p_filt.groupby(df_p_filt['fecha'].dt.isocalendar().week)['PM25'].mean().reset_index()
        fig_ts.add_trace(go.Scatter(x=df_p_week['week'], y=df_p_week['PM25'], name='PM 2.5 Promedio', 
                                    line=dict(color='firebrick', width=2), yaxis='y1'))
    if not df_u_filt.empty:
        df_u_week = df_u_filt.groupby('semana')['Total'].sum().reset_index()
        fig_ts.add_trace(go.Scatter(x=df_u_week['semana'], y=df_u_week['Total'], name='Total Urgencias', 
                                    line=dict(color='royalblue', width=2), yaxis='y2'))
        
    fig_ts.update_layout(
        xaxis_title='Semanas del Año',
        yaxis=dict(title='Nivel PM 2.5 (ug/m3)', titlefont=dict(color='firebrick'), tickfont=dict(color='firebrick')),
        yaxis2=dict(title='Atenciones Urgencia', titlefont=dict(color='royalblue'), tickfont=dict(color='royalblue'),
                    anchor='x', overlaying='y', side='right'),
        margin=dict(t=30, b=30, l=30, r=30),
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)')
    )

    # -------------------------------------------------------------------------
    # 4. Gráfica de Correlación (Scatter)
    # -------------------------------------------------------------------------
    if not df_p_filt.empty and not df_u_filt.empty:
        # Unir por semana para correlacionar
        df_u_week = df_u_filt.groupby('semana')['Total'].sum().reset_index()
        df_p_week = df_p_filt.groupby(df_p_filt['fecha'].dt.isocalendar().week)['PM25'].mean().reset_index()
        df_corr = pd.merge(df_p_week, df_u_week, left_on='week', right_on='semana')
        fig_corr = px.scatter(df_corr, x='PM25', y='Total', trendline="ols", 
                              labels={'PM25': 'Material Particulado 2.5', 'Total': 'Atenciones'},
                              color_discrete_sequence=['#8e44ad'])
    else:
        fig_corr = go.Figure().add_annotation(text="Faltan datos para correlacionar", showarrow=False)
    fig_corr.update_layout(margin=dict(t=20, b=20, l=20, r=20))

    # -------------------------------------------------------------------------
    # 5. Gráfica de Distribución Estadística (Boxplot) - Índice Ocupacional
    # -------------------------------------------------------------------------
    if not df_c_filt.empty and 'INDICE_OCUPACIONAL' in df_c_filt.columns:
        fig_box = px.box(df_c_filt, x='MES', y='INDICE_OCUPACIONAL', color='MES',
                         labels={'MES': 'Mes del Año', 'INDICE_OCUPACIONAL': '% Ocupación'},
                         color_discrete_sequence=px.colors.sequential.Teal_r)
    else:
        fig_box = go.Figure().add_annotation(text="Sin datos de Camas Críticas", showarrow=False)
    fig_box.update_layout(margin=dict(t=20, b=20, l=20, r=20), showlegend=False)

    # -------------------------------------------------------------------------
    # 6. Gráfica de Serie de Tiempo Simple (Camas Críticas)
    # -------------------------------------------------------------------------
    if not df_c_filt.empty and 'DIAS_CAMAS_OCUPADAS' in df_c_filt.columns:
        df_c_grp = df_c_filt.groupby('MES')['DIAS_CAMAS_OCUPADAS'].sum().reset_index()
        fig_camas = px.area(df_c_grp, x='MES', y='DIAS_CAMAS_OCUPADAS', 
                            labels={'MES': 'Mes', 'DIAS_CAMAS_OCUPADAS': 'Días Camas Ocupadas'},
                            color_discrete_sequence=['#e67e22'])
    else:
        fig_camas = go.Figure().add_annotation(text="Sin datos de Camas Críticas", showarrow=False)
    fig_camas.update_layout(margin=dict(t=20, b=20, l=20, r=20))

    return fig_map, fig_comp, fig_ts, fig_corr, fig_box, fig_camas

# ==============================================================================
# 6. EJECUCIÓN DE LA APP
# ==============================================================================
if __name__ == '__main__':
    # jupyter_mode="external" levanta un link local para verlo en pestaña completa.
    # Si quieres verlo en el mismo notebook, cambia a jupyter_mode="inline"
    app.run(debug=True, port=8050, jupyter_mode="external")

Cargando y procesando datos... (Esto puede tomar unos segundos)


NameError: name 'df_urg_2020' is not defined